# Sylber-for-Khmer — Colab T4 pilot run

Runs a small, disk/time-budget-friendly slice of the
[Sylber-Speech-Tokenizer](https://github.com/Pich09/Sylber-Speech-Tokenizer)
pipeline on a single Colab T4 GPU: Step 1 (data prep, streamed subset) →
Step 2 (zero-shot segmentation sanity check) → the CTC probe
(`src/train_ctc.py`), which is the recommended cheap ASR-viability test at
this data scale — see `docs/post-benchmark-roadmap.md` for why (the
full discrete-SLM comparison needs ~1,000+ hours to be trustworthy per the
Sylber papers; the CTC probe is validated at 20-50h, which this notebook
can actually fit).

**Before running**: `Runtime` -> `Change runtime type` -> select **T4 GPU**.

**What this does *not* do**: download the full 1,065h/~495GB corpus, run
the discrete-SLM comparison (`train_slm.py`), or fine-tune Sylber's
backbone (`segmentation.py finetune`) — those need more disk/time than a
free/Pro Colab session reasonably offers. This notebook is for a quick
pilot/sanity run; do the full runs on the RTX 4070 machine per
`docs/run-roadmap.md`.

## 0. Check the GPU

In [ ]:
!nvidia-smi

## 1. Clone the repo and install dependencies

Colab already ships a CUDA-enabled `torch` build, so `requirements.txt` is
installed as-is (its `torch>=2.1.0` pin is already satisfied) rather than
reinstalling torch from a specific CUDA index, which is only needed on a
machine without a CUDA-matched wheel already present.

In [ ]:
!git clone https://github.com/Pich09/Sylber-Speech-Tokenizer.git
%cd Sylber-Speech-Tokenizer

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch
print("torch", torch.__version__, "cuda available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU visible — check Runtime > Change runtime type > T4 GPU"

### Patch: `sylber`/`torchtyping` incompatibility

`docs/path-a-encoder-comparison.md`'s "Pipeline notes" section records that
the unmaintained `torchtyping` dependency crashes on newer torch
(`Cannot subclass _TensorBase directly`) when `sylber` imports its unused
`SegmentSynthesis` class. This patches the installed package to skip that
import defensively — a no-op if your torch/torchtyping combination doesn't
hit the bug.

In [ ]:
import re
from pathlib import Path

import sylber

init_path = Path(sylber.__file__)
src = init_path.read_text()

if "SegmentSynthesis" in src and "# patched-by-colab-notebook" not in src:
    patched = re.sub(
        r"^(from .*SegmentSynthesis.*|.*import SegmentSynthesis.*)$",
        r"try:\n    \1\nexcept Exception as _e:  # patched-by-colab-notebook: unused import, torchtyping incompatible with newer torch\n    print('sylber: skipped SegmentSynthesis import (', _e, ')')",
        src,
        flags=re.MULTILINE,
    )
    if patched != src:
        init_path.write_text(patched)
        print("Patched", init_path)
    else:
        print("No matching SegmentSynthesis import line found — leaving sylber/__init__.py as-is.")
else:
    print("Already patched or no SegmentSynthesis import present — nothing to do.")

## 2. (Optional) Save outputs to Google Drive

Colab's local disk is wiped when the runtime disconnects. Uncomment this
cell if you want the manifest/results/checkpoints copied to Drive at the
end of the notebook instead of just living in the ephemeral VM.

In [ ]:
USE_DRIVE = False  # flip to True to mount Drive and persist outputs at the end

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

## 3. Step 1 — data prep (small streamed subset)

`prepare_data.py --max-samples N` streams only the first N examples from
the Hugging Face dataset instead of downloading the full corpus (this flag
was added specifically to make a Colab-scale run possible — the default
behavior downloads the entire ~495GB/1,065h corpus, which doesn't fit
here). Utterances in this corpus average ~8.5s, so `MAX_SAMPLES=800`
gives roughly ~1.9h of audio — enough for a zero-shot sanity check and a
CTC-probe train/val split. Raise it (e.g. to 3000-5000 for ~7-12h) if you
have the disk/time budget and want a firmer CTC-probe read.

In [ ]:
MAX_SAMPLES = 800
DATASET = "DDD-Cambodia/khmer-speech-dataset"
MANIFEST_NAME = "khmer_colab_pilot_manifest.csv"

!python data/preprocessing/prepare_data.py \
    --dataset {DATASET} \
    --out data/khmer_colab_pilot \
    --max-samples {MAX_SAMPLES} \
    --manifest-name {MANIFEST_NAME}

In [ ]:
import pandas as pd

MANIFEST_PATH = f"data/preprocessing/manifests/{MANIFEST_NAME}"
df = pd.read_csv(MANIFEST_PATH)
print(f"{len(df)} utterances, {df['duration_sec'].sum() / 3600:.2f}h total")
print(df["split"].value_counts())
df.head()

## 4. Sanity-check `sylber`'s output shape on one file

README/roadmap both flag this as the one place a newer/older `sylber`
release could differ from what this code assumes (a `segment_features`
key). Worth checking before running the full pilot below.

In [ ]:
from sylber import Segmenter

_seg = Segmenter(model_ckpt="sylber")
_out = _seg(df.iloc[0]["path"], in_second=True)
print("Output keys:", list(_out.keys()) if isinstance(_out, dict) else type(_out))
del _seg, _out

## 5. Step 2 — zero-shot segmentation eval

Expected token rate is 4-5 Hz; the script flags PASS/WARNING automatically.

In [ ]:
!python src/segmentation.py eval \
    --manifest {MANIFEST_PATH} \
    --n-samples 50 \
    --out results/zero_shot_evaluation.txt

print(open("results/zero_shot_evaluation.txt").read())

## 6. CTC probe — the recommended cheap ASR-viability test

Trains a small supervised CTC decoder on top of each frozen encoder's
features and reports character error rate (CER) on the val split. This is
the test validated at this data scale by Sylber 2.0's own "Low-Resource
ASR" experiments (20-50h per language) — unlike the discrete-SLM
comparison, it doesn't need hundreds/thousands of hours to be meaningful.

Runs Sylber first (the encoder you actually care about); HuBERT is
included as a quick reference point since it's the cheaper of the two
baselines to run on a T4. Whisper is left commented out — add it back if
you have the time budget (its 30s-chunked forward pass is the slowest of
the three).

In [ ]:
!python src/train_ctc.py train \
    --manifest {MANIFEST_PATH} \
    --encoder sylber \
    --epochs 15

In [ ]:
!python src/train_ctc.py train \
    --manifest {MANIFEST_PATH} \
    --encoder hubert \
    --checkpoint facebook/hubert-base-ls960 \
    --epochs 15

In [ ]:
# Uncomment to also run Whisper (slower — 30s-chunked forward pass):
# !python src/train_ctc.py train \
#     --manifest {MANIFEST_PATH} \
#     --encoder whisper \
#     --checkpoint openai/whisper-base \
#     --epochs 15

## 7. Compare results

In [ ]:
import json
from pathlib import Path

for p in sorted(Path("results/downstream_eval").glob("ctc_probe_*.json")):
    r = json.loads(p.read_text())
    print(f"{r['encoder']:8s} final_val_cer={r['final_val_cer']}  n_train={r['n_train']}  n_val={r['n_val']}")

Remember this is a small pilot (`MAX_SAMPLES` utterances, few epochs) —
read it as a directional signal, not a final result. If Sylber's CER is
competitive here, `docs/post-benchmark-roadmap.md`'s Option A section has
the decision tree for what to do next (proceed to Path B's full fine-tune
with confidence). If it's clearly worse, that doc's backup ladder covers
what to try before writing Sylber off (fine-tune the backbone before
judging zero-shot, watch for a Sylber 2.0 checkpoint release, etc.) — none
of which need repeating on Colab; do those on the RTX 4070 machine.

## 8. (Optional) Persist outputs to Drive

In [ ]:
if USE_DRIVE:
    import shutil
    dest = Path("/content/drive/MyDrive/sylber-khmer-colab-pilot")
    dest.mkdir(parents=True, exist_ok=True)
    for d in ["data/preprocessing/manifests", "results/downstream_eval", "models", "logs"]:
        src_dir = Path(d)
        if src_dir.exists():
            shutil.copytree(src_dir, dest / d, dirs_exist_ok=True)
    print("Copied outputs to", dest)
else:
    print("USE_DRIVE is False — outputs will be lost when this runtime disconnects.")